In [2]:
from rag_helper import RAGBase
from ingest import load_course_data, build_index

In [4]:
import os
from getpass import getpass
from dotenv import load_dotenv
load_dotenv()

from openai import OpenAI
# This opens a secure input box to paste your key
os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API Key: ")
openai_client = OpenAI()

In [5]:
documents = load_course_data()
index = build_index(documents)

In [6]:
assistant = RAGBase(index, openai_client)

In [ ]:
assistant.rag('How does the agentic loop keep calling the model until it stops?')

('The loop keeps calling the model with a `while True` loop and stops only when the model returns **no function calls**.\n\nIn the code:\n\n- call the model\n- check each item in `response.output`\n- if there’s a `function_call`, run the tool, append the tool result, and set `has_function_calls = True`\n- if there are **no** function calls in that turn, `break`\n\nSo the exit condition is:\n\n```python\nif has_function_calls == False:\n    break\n```\n\nIn short: **keep looping while the model asks for tools; stop when it gives a final message without any tool calls.**',
 ResponseUsage(input_tokens=7114, input_tokens_details=InputTokensDetails(cache_write_tokens=0, cached_tokens=0), output_tokens=138, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=7252))

In [8]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

In [15]:
#print number of chunks
print(f'Number of chunks: {len(chunks)}')


Number of chunks: 295


In [16]:
chunk_index = build_index(chunks)
assistant.index = chunk_index

In [17]:
assistant.rag('How does the agentic loop keep calling the model until it stops?')

('The loop keeps calling the model inside a `while True` loop.\n\nEach turn:\n\n1. The model is called with the current `messages`.\n2. If the response contains any `function_call` items, your code runs them and adds the tool outputs back into `messages`.\n3. A flag `has_function_calls` is set to `True` if any tool was called.\n4. If there were no function calls in that turn, the loop breaks.\n\nSo the stopping condition is:\n\n- **no function calls in the model’s response** → **done**\n\nIn code, that’s this part:\n\n```python\nif has_function_calls == False:\n    break\n```\n\nIf you want, I can also explain how the message history makes the next model call aware of previous tool results.',
 ResponseUsage(input_tokens=2294, input_tokens_details=InputTokensDetails(cache_write_tokens=0, cached_tokens=0), output_tokens=164, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=2458))